# ICS 591 Team 2 — Evaluation 2: the **fine-tuned model (v1)** on the local test set

**The model.** ViT-S/16 from Stage 1, fine-tuned in Stage 2c on RealWaste + 304 local images, with two heads:
material (plastic / non-plastic / discard) and cleanliness (clean / contaminated). It gives one of four decisions:
accept, reject (dirty), reject (wrong type) or retake, plus a calibrated confidence.

**What this notebook reports on the 113 local test photos** (never used in training):
1. Headline numbers with 95% confidence intervals, and a check that they match the Stage 2c run.
2. Per-decision precision and recall, the confusion matrix, and the two money errors (FAR, FRR).
3. Calibration on local photos (is "90% sure" really right 90% of the time?).
4. Side-by-side comparison with the baseline (needs notebook 08 to have run).
5. Pictures of the mistakes that cost money, for the slides.

The confidence threshold for handing photos to a supervisor is studied separately in notebook 10.

**How to run:** Runtime → Run all. No GPU needed.

In [ ]:
# 1. Setup: paths, the local TEST set, helpers. A GPU is optional (113 images also run on CPU in a few minutes).
!pip -q install timm
import os, json, shutil, pathlib, random, hashlib
import numpy as np, pandas as pd, torch, timm, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torchvision import transforms as T
from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive')
LOCAL_OUT = DRIVE / 'ics591_local_data'             # Stage 2a/2b: images + label files
IMG_DRIVE = LOCAL_OUT / 'raw' / 'images'
S1_OUT = DRIVE / 'ics591_model_selection' / 'realwaste'   # Stage 1 outputs
STAGE1_CKPT = S1_OUT / 'ViT-S_16.pt'                 # baseline: Stage 1 ViT, RealWaste only
FT_OUT = DRIVE / 'ics591_finetune'                   # Stage 2c outputs
V1_CKPT, V1_CFG = FT_OUT / 'verifier_v1.pt', FT_OUT / 'verifier_v1_config.json'
EVAL_OUT = DRIVE / 'ics591_evaluation'; EVAL_OUT.mkdir(exist_ok=True)   # everything this notebook saves goes here
TIMM_ID = 'vit_small_patch16_224.augreg_in21k_ft_in1k'
for p in [IMG_DRIVE, LOCAL_OUT / 'al_queue.csv']: assert p.exists(), f'missing: {p}'

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Running on', DEV)

MATERIAL = ['plastic', 'non_plastic', 'discard']
CLEAN = ['clean', 'contaminated']
DECISIONS = ['accept', 'reject_contaminated', 'reject_wrong_type', 'retake_photo']
NICE = {'accept': 'Accept', 'reject_contaminated': 'Reject: dirty', 'reject_wrong_type': 'Reject: wrong type',
        'retake_photo': 'Retake'}

# --- label files (exactly the same logic as notebook 05, so the test set is the same 113 images)
TEAM = ['nana', 'kelvin', 'cajetan']
QUEUE, AUTO = LOCAL_OUT / 'al_queue.csv', LOCAL_OUT / 'al_auto_labels.csv'
def REV(p): return LOCAL_OUT / f'al_reviews_{p}.csv'
def to_model_label(l): return 'discard' if str(l).startswith('discard') else l
def load_reviews():
    parts = [pd.read_csv(REV(p)) for p in TEAM if REV(p).exists()]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['file', 'label', 'pool', 'labeler', 'ts'])
def test_files():
    q = pd.read_csv(QUEUE); return set(q.loc[q['pool'] == 'test', 'file'])
def test_labels():
    r = load_reviews(); r = r[r['pool'] == 'test'].sort_values('ts').drop_duplicates(['file', 'labeler'], keep='last')
    out = []
    for f, g in r.groupby('file'):
        vc = g['label'].map(to_model_label).value_counts()
        if len(vc) == 1 or vc.iloc[0] > vc.iloc[1]: out.append({'file': f, 'label': vc.index[0]})
    return pd.DataFrame(out, columns=['file', 'label'])
def current_labels():
    r = load_reviews(); r = r[r['pool'] != 'test'].sort_values('ts').drop_duplicates('file', keep='last')
    human = pd.DataFrame({'file': r['file'], 'label': r['label'].map(to_model_label), 'source': 'human'})
    auto = pd.read_csv(AUTO) if AUTO.exists() else pd.DataFrame(columns=['file', 'label'])
    auto = pd.DataFrame({'file': auto['file'], 'label': auto['label'], 'source': 'model'})
    auto = auto[~auto['file'].isin(human['file'])]
    lab = pd.concat([human, auto], ignore_index=True)
    return lab[~lab['file'].isin(test_files())].reset_index(drop=True)
LABEL_TO_DECISION = {'plastic_clean': 0, 'plastic_contaminated': 1, 'non_plastic': 2, 'discard': 3}
LABEL_TO_MATERIAL = {'plastic_clean': 0, 'plastic_contaminated': 0, 'non_plastic': 1, 'discard': 2}

loc_te = test_labels().sort_values('file').reset_index(drop=True)
print('Local TEST set:', len(loc_te), 'images', loc_te['label'].value_counts().to_dict())
if len(loc_te) != 113: print('WARNING: expected 113 test images (as in Stage 2c). Have label files changed since?')
y_dec = loc_te['label'].map(LABEL_TO_DECISION).values     # true decision: 0 accept, 1 dirty, 2 wrong type, 3 retake
y_mat = loc_te['label'].map(LABEL_TO_MATERIAL).values     # true material: 0 plastic, 1 non-plastic, 2 discard

CACHE = pathlib.Path('/content/img_cache'); CACHE.mkdir(exist_ok=True)
for f in loc_te['file']:
    if not (CACHE / f).exists(): shutil.copy(IMG_DRIVE / f, CACHE / f)

_cfg = timm.data.resolve_data_config({}, model=timm.create_model(TIMM_ID, pretrained=False))
EVAL_TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(_cfg['mean'], _cfg['std'])])

@torch.no_grad()
def run(model, files, batch=32):
    """Runs a model over image files; returns a list with one numpy array per model output."""
    model.eval(); outs = None
    for i in range(0, len(files), batch):
        x = torch.stack([EVAL_TF(Image.open(CACHE / f).convert('RGB')) for f in files[i:i + batch]]).to(DEV)
        o = model(x); o = o if isinstance(o, tuple) else (o,)
        o = [t.float().softmax(1).cpu().numpy() for t in o]
        outs = o if outs is None else [np.vstack([a, b]) for a, b in zip(outs, o)]
    return outs

def ci(correct, n_boot=2000):
    """95% bootstrap interval for a mean of 0/1 values (resample the test images 2000 times)."""
    v = np.asarray(correct, dtype=float); rng = np.random.default_rng(SEED)
    b = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n_boot)]
    return [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]
def pct(x): return f'{100 * x:.1f}%'
def show_examples(idx, title, captions, path, cols=5):
    """Picture grid for slides."""
    idx = list(idx)[:15]
    if not idx: print('no examples'); return
    rows = int(np.ceil(len(idx) / cols)); fig, axs = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.6 * rows), squeeze=False)
    for a in axs.flat: a.axis('off')
    for a, i in zip(axs.flat, idx):
        a.imshow(Image.open(CACHE / loc_te['file'][i]).convert('RGB')); a.set_title(captions[i], fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=130); plt.show()

In [ ]:
# Load the fine-tuned model v1 (the same weights that were converted to ONNX and deployed)
class VerifierV1(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone; d = backbone.num_features
        self.material = nn.Linear(d, len(MATERIAL)); self.clean = nn.Linear(d, len(CLEAN))
    def forward(self, x):
        f = self.backbone(x); return self.material(f), self.clean(f)

cfg = json.loads(V1_CFG.read_text())
bb = timm.create_model(TIMM_ID, pretrained=False, num_classes=0)
v1 = VerifierV1(bb); v1.load_state_dict(torch.load(V1_CKPT, map_location='cpu')); v1 = v1.to(DEV)
T_MAT, T_CLEAN = cfg['temperature_material'], cfg['temperature_clean']
print('weights sha256:', hashlib.sha256(V1_CKPT.read_bytes()).hexdigest()[:16], '| temperatures', round(T_MAT, 3), round(T_CLEAN, 3),
      '| classifier threshold in config', round(cfg['auto_decision_threshold'], 3))

def apply_T(p, t):
    z = np.log(np.clip(p, 1e-12, 1)) / t; z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def decide(pm, pc):
    """Same rule as notebook 05 and the deployed service (decision.py)."""
    mat = pm.argmax(1); cl = pc.argmax(1)
    dec = np.select([mat == 2, mat == 1, (mat == 0) & (cl == 1)], [3, 2, 1], default=0)
    conf = np.where(mat == 0, pm.max(1) * pc.max(1), pm.max(1))
    return dec, conf
def v1_predict(files):
    pm, pc = run(v1, list(files)); pm, pc = apply_T(pm, T_MAT), apply_T(pc, T_CLEAN)
    dec, conf = decide(pm, pc); return dec, conf, pm, pc

dec, conf, pm, pc = v1_predict(loc_te['file'])
print('done:', len(dec), 'photos')

In [ ]:
# 2. Headline numbers
from sklearn.metrics import f1_score
ok = dec == y_dec
should_pay, pays = y_dec == 0, dec == 0
plast = y_mat == 0; w = y_mat != 2
R = {'model': 'v1: ViT-S/16 fine-tuned, two heads', 'local_test_n': int(len(loc_te))}
R['decision_accuracy_4way'] = float(ok.mean()); R['decision_accuracy_4way_95ci'] = ci(ok)
R['decision_macro_f1_4way'] = float(f1_score(y_dec, dec, average='macro', labels=range(4), zero_division=0))
R['material_accuracy'] = float((pm.argmax(1) == y_mat).mean())
R['cleanliness_accuracy_on_plastic'] = float((pc[plast].argmax(1) == np.where(y_dec[plast] == 0, 0, 1)).mean())
R['cleanliness_n'] = int(plast.sum())
R['wrong_payments'] = int((pays & ~should_pay).sum())
R['false_accept_rate'] = float((pays & ~should_pay).sum() / (~should_pay).sum())
R['honest_missed'] = int((~pays & should_pay).sum())
R['false_reject_rate'] = float((~pays & should_pay).sum() / should_pay.sum())
R['wrong_payments_by_true_label'] = loc_te['label'][pays & ~should_pay].value_counts().to_dict()
ok_pv = (pm[w][:, :2].argmax(1) == y_mat[w])
R['plastic_vs_non_accuracy'] = float(ok_pv.mean()); R['plastic_vs_non_accuracy_95ci'] = ci(ok_pv); R['plastic_vs_non_n'] = int(w.sum())

prev = FT_OUT / 'v1_test_results.json'
if prev.exists():
    S = json.loads(prev.read_text())
    same = abs(S['decision_accuracy_all'] - R['decision_accuracy_4way']) < 1e-6
    print('Same as the Stage 2c run:', 'YES' if same else f"NO (Stage 2c {pct(S['decision_accuracy_all'])}, now {pct(R['decision_accuracy_4way'])})")

(EVAL_OUT / 'v1_results.json').write_text(json.dumps(R, indent=2))
pd.DataFrame({'file': loc_te['file'], 'true_label': loc_te['label'], 'v1_decision': [DECISIONS[d] for d in dec],
              'confidence': conf.round(4), 'p_plastic': pm[:, 0].round(4), 'p_non_plastic': pm[:, 1].round(4),
              'p_discard': pm[:, 2].round(4), 'p_clean': pc[:, 0].round(4)}).to_csv(EVAL_OUT / 'v1_predictions.csv', index=False)
pd.DataFrame([
    ['Four app decisions correct', f"{pct(R['decision_accuracy_4way'])}  (95% CI {pct(R['decision_accuracy_4way_95ci'][0])}-{pct(R['decision_accuracy_4way_95ci'][1])})"],
    ['Macro-F1 (four decisions)', f"{R['decision_macro_f1_4way']:.3f}"],
    ['Material correct (plastic / non-plastic / discard)', pct(R['material_accuracy'])],
    [f"Clean vs dirty correct (plastic photos, n = {R['cleanliness_n']})", pct(R['cleanliness_accuracy_on_plastic'])],
    ['Wrong payments, FAR', f"{R['wrong_payments']} of {int((~should_pay).sum())}  ({pct(R['false_accept_rate'])})  {R['wrong_payments_by_true_label']}"],
    ['Honest households missed, FRR', f"{R['honest_missed']} of {int(should_pay.sum())}  ({pct(R['false_reject_rate'])})"],
    [f"Plastic vs non-plastic (waste photos, n = {R['plastic_vs_non_n']})", pct(R['plastic_vs_non_accuracy'])],
], columns=['Measure', 'Fine-tuned v1'])

In [ ]:
# 3. Per decision: precision (when it says X, how often right) and recall (of all true X, how many found)
from sklearn.metrics import classification_report, confusion_matrix
rep = pd.DataFrame(classification_report(y_dec, dec, labels=range(4), target_names=[NICE[d] for d in DECISIONS],
                                         output_dict=True, zero_division=0)).T.round(3)
rep.to_csv(EVAL_OUT / 'v1_per_decision_report.csv'); display(rep)

cm = confusion_matrix(y_dec, dec, labels=range(4))
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for a, m, title, labels in [(ax[0], cm, 'Four decisions (rows: truth)', [NICE[d] for d in DECISIONS]),
                            (ax[1], np.array([[(pays & should_pay).sum(), (~pays & should_pay).sum()],
                                              [(pays & ~should_pay).sum(), (~pays & ~should_pay).sum()]]),
                             'Paid or not (rows: should it be paid?)', ['Paid', 'Not paid'])]:
    a.imshow(m, cmap='Greys', vmin=0, vmax=m.max() * 1.6)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]): a.text(j, i, int(m[i, j]), ha='center', va='center', fontsize=13)
    a.set_xticks(range(len(labels)), labels, rotation=20); a.set_yticks(range(len(labels)), labels)
    a.set_xlabel('model said'); a.set_title(title)
plt.tight_layout(); plt.savefig(EVAL_OUT / 'v1_confusion.png', dpi=150); plt.show()

In [ ]:
# 4. Calibration on LOCAL photos: when the model says "80% sure", is it right about 80% of the time?
bins = np.array([0, .5, .7, .8, .9, 1.0001])
rows = []
for lo, hi in zip(bins[:-1], bins[1:]):
    s = (conf >= lo) & (conf < hi)
    if s.any(): rows.append({'confidence': f'{lo:.1f}-{min(hi, 1):.1f}', 'photos': int(s.sum()),
                             'mean confidence': conf[s].mean().round(3), 'actually right': ok[s].mean().round(3)})
cal = pd.DataFrame(rows)
ece = float(sum(r['photos'] * abs(r['mean confidence'] - r['actually right']) for r in rows) / len(conf))
R['ece_local'] = ece; (EVAL_OUT / 'v1_results.json').write_text(json.dumps(R, indent=2))
cal.to_csv(EVAL_OUT / 'v1_calibration_table.csv', index=False)
plt.figure(figsize=(5, 4.5)); plt.plot([0, 1], [0, 1], 'k--', lw=1, label='perfect')
plt.plot(cal['mean confidence'], cal['actually right'], 'ko-', label='v1 (local test)')
for _, r in cal.iterrows(): plt.annotate(f"n={r['photos']}", (r['mean confidence'], r['actually right']), textcoords='offset points', xytext=(5, -12), fontsize=8)
plt.xlabel('confidence the model gave'); plt.ylabel('share actually right'); plt.title(f'Calibration on local photos (ECE {ece:.3f})')
plt.legend(); plt.tight_layout(); plt.savefig(EVAL_OUT / 'v1_calibration.png', dpi=150); plt.show()
cal

In [ ]:
# 5. Baseline vs fine-tuned, side by side (reads notebook 08's results)
bp = EVAL_OUT / 'baseline_results.json'
if not bp.exists():
    print('Run notebook 08 first to get the baseline column.')
else:
    B = json.loads(bp.read_text())
    comp = pd.DataFrame([
        ['Plastic vs non-plastic (waste photos)', pct(B['plastic_vs_non_accuracy']), pct(R['plastic_vs_non_accuracy'])],
        ['Wrong payments, FAR', f"{B['wrong_payments']} ({pct(B['false_accept_rate'])})", f"{R['wrong_payments']} ({pct(R['false_accept_rate'])})"],
        ['Honest households missed, FRR', f"{B['honest_missed']} ({pct(B['false_reject_rate'])})", f"{R['honest_missed']} ({pct(R['false_reject_rate'])})"],
        ['Four app decisions correct', pct(B['decision_accuracy_4way']), pct(R['decision_accuracy_4way'])],
        ['Can say "dirty"?', 'no', f"yes ({pct(R['cleanliness_accuracy_on_plastic'])} right)"],
        ['Can say "retake"?', 'no', 'yes'],
    ], columns=['Measure (113 local test photos)', 'Baseline (RealWaste only)', 'Fine-tuned v1'])
    comp.to_csv(EVAL_OUT / 'baseline_vs_v1.csv', index=False)
    print('Trivial floors: always non-plastic', pct(B['trivial_always_non_plastic_on_waste_photos']),
          '| most common decision', pct(B['trivial_majority_decision_4way']))
    display(comp)

In [ ]:
# 6. The mistakes that cost money (v1 would pay, but should not), and clean plastics it did not pay
cap = {i: f"truth: {NICE[DECISIONS[y_dec[i]]]}\nv1: {NICE[DECISIONS[dec[i]]]} ({conf[i]:.2f})" for i in range(len(loc_te))}
show_examples(np.where(pays & ~should_pay)[0], 'v1 would pay, but should not', cap, EVAL_OUT / 'v1_wrong_payments.png')
show_examples(np.where(~pays & should_pay)[0], 'Clean plastic v1 did not pay', cap, EVAL_OUT / 'v1_missed_clean.png')
# the biggest confusion: non-plastic called "dirty plastic"
show_examples(np.where((y_dec == 2) & (dec == 1))[0], 'Non-plastic called "dirty plastic" (sample)', cap, EVAL_OUT / 'v1_nonplastic_called_dirty.png')

## How to read this
- **FAR and FRR** are the money results: wrong payments lose money, missed honest households lose trust.
- **Four app decisions** is the full job. The baseline cannot do it at all for dirty and retake photos.
- **Calibration**: if the dots sit near the diagonal, the confidence can be used to decide when to hand a photo to a supervisor (notebook 10).
- Files are in `MyDrive/ics591_evaluation/`: `v1_results.json`, `v1_predictions.csv`, tables and pictures.